In [1]:
# =======================
# 1. Imports
# =======================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor
import optuna
import mlflow
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
import mlflow.sklearn

c:\Users\sheri\Desktop\machinlearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================
# 2. Load datasets
# =============================
train_df = pd.read_csv(r'C:\Users\sheri\Desktop\machinlearning\data\processed_data\feature_engineered_train.csv')
eval_df = pd.read_csv(r'C:\Users\sheri\Desktop\machinlearning\data\processed_data\featured_engineered_eval.csv')

# Define target & features
target = "AEP_MW"
X_train, y_train  = train_df.drop(columns=[target]), train_df[target]

X_eval, y_eval = eval_df.drop(columns=[target]), eval_df[target]

print("Train Shape:", X_train.shape)
print("Eval Shape:", X_eval.shape)

Train Shape: (62920, 4)
Eval Shape: (34923, 4)


In [3]:
# ======================================================
# 3. Define OPtuna objective function with MLFlow
# ======================================================
def objective(trial):
    params = {
    "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
    "subsample": trial.suggest_float("subsample", 0.5, 1.0),
    "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
    "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 8),
    "min_weight_fraction_leaf": trial.suggest_int("min_weight_fraction_leaf", 0.0, 0.5),
    "max_depth": trial.suggest_int("max_depth", 3, 10),
    "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0.0, 6),
    "random_state": 42,
    "alpha": trial.suggest_float("alpha", 0.0, 1.0),
    "tol": trial.suggest_float("tol", 0.0, 7),
    "ccp_alpha": trial.suggest_float("ccp_alpha", 0.0, 8)
    }

    with mlflow.start_run(nested=True):
        model = GradientBoostingRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # log hyperparameter + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    return rmse

    

In [4]:
# =====================================
# 4. Run Optuna study with MLFlow
# =====================================
# Force MLFlow to always use the root project mlrun folder
#mlflow.set_tracking_uri(r'C:\Users\sheri\Desktop\machinlearning\mlruns')
mlflow.set_tracking_uri("file:///C:/Users/sheri/Desktop/machinlearning/mlruns")
mlflow.set_experiment("GradientBoostingRegressor_energy")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

print("Best params:", study.best_params)

[I 2026-07-28 18:10:08,866] A new study created in memory with name: no-name-0f07f650-f3cb-4c83-9c9b-ceb6385cb324
[I 2026-07-28 18:10:21,242] Trial 0 finished with value: 1826.0913552827603 and parameters: {'n_estimators': 637, 'learning_rate': 0.02428261547029146, 'subsample': 0.5075876258183779, 'min_samples_split': 11, 'min_samples_leaf': 8, 'min_weight_fraction_leaf': 0, 'max_depth': 6, 'min_impurity_decrease': 4.498317200347019, 'alpha': 0.822402758918227, 'tol': 6.887921390455897, 'ccp_alpha': 6.656513930285807}. Best is trial 0 with value: 1826.0913552827603.
[I 2026-07-28 18:10:27,691] Trial 1 finished with value: 1761.354618218889 and parameters: {'n_estimators': 285, 'learning_rate': 0.05403744624913964, 'subsample': 0.8022660235205457, 'min_samples_split': 9, 'min_samples_leaf': 4, 'min_weight_fraction_leaf': 0, 'max_depth': 5, 'min_impurity_decrease': 5.91883075064555, 'alpha': 0.13923067111978227, 'tol': 0.38007376198547915, 'ccp_alpha': 6.697829770304013}. Best is trial 1

Best params: {'n_estimators': 341, 'learning_rate': 0.01349178732093854, 'subsample': 0.8211712379220841, 'min_samples_split': 14, 'min_samples_leaf': 3, 'min_weight_fraction_leaf': 0, 'max_depth': 7, 'min_impurity_decrease': 3.467152681871565, 'alpha': 0.2481895956604594, 'tol': 2.8276795019328294, 'ccp_alpha': 2.769594515355501}


In [5]:
# ============================================================
# 5. Train final model with best params and log to MLFlow
# ============================================================
best_params = study.best_trial.params
best_model = GradientBoostingRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

# log final model
with mlflow.start_run(run_name="best_GradientBoostingRegressor_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.sklearn.log_model(sk_model=best_model, name="model") # or artifact_path="model" for MLflow 2.x

Final tuned model performance:
MAE: 1361.6625285107639
RMSE: 1728.982510931817
R2: 0.5079504018737034
